Homework 10
==========
<link rel="stylesheet" href="/Users/pedrofatecha/GitHub/CodePF/ISYE6501_IntrotoAnalyticsModeling/02_Script/02_1_Function/styles.css">

>📌 Question 14.1 </br>
> The breast cancer data set breast-cancer-wisconsin.data.txt from https://archive.ics.uci.edu/dataset/15/breast+cancer+wisconsin+original  (description available at the same URL) has missing values.
>1.	Use the mean/mode imputation method to impute values for the missing data.
>2.	Use regression to impute values for the missing data.
>3.	Use regression with perturbation to impute values for the missing data.
>4.	(Optional) Compare the results and quality of classification models (e.g., SVM, KNN) build using</br>
>(1) the data sets from questions 1,2,3;</br>
>(2) the data that remains after data points with missing values are removed; and</br>
>(3) the data set when a binary variable is introduced to indicate missing values.</br>


</br>

In [78]:
library("kernlab")
library("kknn")
library("tidyr")
library("dplyr")
library("ggplot2")
library("plotly")
library("combinat")
library("ggplot2")
library("rpart")
library("randomForest")
library("rpart.plot")
library("caret")
library(data.table)
library(mltools)
library(glmnet)
library(truncnorm)
####Functions
read_data<-function(data.path = character(), is_header= TRUE,delim = "\t"){
    # Reads the data from the given path and returns a data frame. In this case, the data is tab separated and has a header.
    data <- read.csv(data.path,header = is_header,sep = delim)
    return(data)

}


impute_data<-function(data, method = "mean"){
    # Imputes the missing values in the data using the specified method. The default method is mean imputation.
    if(method == "mean"){
        for(i in 1:ncol(data)){
            data[is.na(data[,i]), i] <- mean(data[,i], na.rm = TRUE)
        }
    } else if(method == "median"){
        for(i in 1:ncol(data)){
            data[is.na(data[,i]), i] <- median(data[,i], na.rm = TRUE)
        }
    } else if(method == "mode"){
        for(i in 1:ncol(data)){
            mode_value <- as.numeric(names(sort(table(data[,i]), decreasing = TRUE)[1]))
            data[is.na(data[,i]), i] <- mode_value
        }
    }
    return(data)
}



In [92]:
Cancer_wisconsin<-read_data("/Users/pedrofatecha/GitHub/CodePF/ISYE6501_IntrotoAnalyticsModeling/01_Data/Homework10_ISYE6501/breast-cancer-wisconsin.data.txt",is_header = FALSE,delim = ",")

#Cancer_wisconsin[apply(Cancer_wisconsin, 1, function(x) any(x == "?")),] # Check for missing values

number_oflines_missing<-nrow(Cancer_wisconsin[apply(Cancer_wisconsin, 1, function(x) any(x == "?")),])
number_without_missing<-nrow(Cancer_wisconsin)


cat("Number of Lines with missing value: ",number_oflines_missing," Column V7, ","this represent",round((number_oflines_missing/number_without_missing)*100,2),"% of the data")
Cancer_wisconsin[Cancer_wisconsin[,"V7"] == "?","V7"] <- NA # Replace "?" with NA in column V7

Cancer_wisconsin$V7<-as.numeric(Cancer_wisconsin$V7) # Convert column V7 to numeric



Number of Lines with missing value:  16  Column V7,  this represent 2.29 % of the data

In [80]:



Clean_dataset_Modeimputed<-impute_data(data = Cancer_wisconsin, method = "mode") # Impute the missing values using mode imputation


View(Clean_dataset_Modeimputed[is.na(Clean_dataset_Modeimputed$V7),])


Clean_dataset_meanimputed<-impute_data(data = Cancer_wisconsin) # Impute the missing values using mode imputation

View(Clean_dataset_meanimputed[is.na(Clean_dataset_meanimputed$V7),])




V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11
<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>


V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11
<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>


In [81]:
######Imputation with linear regression 


imputed_with_lm<-Cancer_wisconsin # 1. Create a copy of the original dataset to impute with linear regression

fit <- lm(V7 ~ V3 + V4 + V5 + V8 + V9 + V11, data = Cancer_wisconsin, subset = !is.na(V7)) # 2. fitting a linear regression model using the rows without missing values in column V7

summary(fit)

index<-which(x = apply(Cancer_wisconsin, 1, function(x) any(is.na(x))), arr.ind = TRUE) # 3. Get the indices of the rows with missing values

predicted_values <- predict(fit, newdata = imputed_with_lm[is.na(imputed_with_lm$V7),]) # 4. Predict the missing values using the fitted model

imputed_with_lm[is.na(imputed_with_lm$V7), "V7"] <- as.integer(predicted_values) # 5. Impute the missing values in column V7 with the predicted values from the linear regression model

View(imputed_with_lm[index,])


summary(imputed_with_lm[index,])



Call:
lm(formula = V7 ~ V3 + V4 + V5 + V8 + V9 + V11, data = Cancer_wisconsin, 
    subset = !is.na(V7))

Residuals:
    Min      1Q  Median      3Q     Max 
-7.9685 -0.4130 -0.2560  0.8506  8.5870 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -4.28748    0.30423 -14.093  < 2e-16 ***
V3          -0.16300    0.06499  -2.508  0.01238 *  
V4           0.18770    0.06497   2.889  0.00399 ** 
V5           0.21369    0.04068   5.253 2.01e-07 ***
V8           0.15697    0.05301   2.961  0.00317 ** 
V9          -0.09283    0.03907  -2.376  0.01779 *  
V11          2.54202    0.16391  15.509  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1.998 on 676 degrees of freedom
Multiple R-squared:  0.702,	Adjusted R-squared:  0.6993 
F-statistic: 265.4 on 6 and 676 DF,  p-value: < 2.2e-16


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11
,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>
24,1057013,8,4,5,1,2,7,7,3,1,4
41,1096800,6,6,6,9,6,3,7,8,1,2
140,1183246,1,1,1,1,1,1,2,1,1,2
146,1184840,1,1,3,1,2,1,2,1,1,2
159,1193683,1,1,2,1,3,1,1,1,1,2
165,1197510,5,1,1,1,2,1,3,1,1,2
236,1241232,3,1,4,1,2,1,3,1,1,2
250,169356,3,1,1,1,2,1,3,1,1,2
276,432809,3,1,3,1,2,1,2,1,1,2


       V1                V2              V3              V4       
 Min.   :  61634   Min.   :1.000   Min.   :1.000   Min.   :1.000  
 1st Qu.: 595517   1st Qu.:1.000   1st Qu.:1.000   1st Qu.:1.000  
 Median :1057040   Median :3.000   Median :1.000   Median :2.500  
 Mean   : 857578   Mean   :3.375   Mean   :2.438   Mean   :2.875  
 3rd Qu.:1187051   3rd Qu.:5.000   3rd Qu.:4.000   3rd Qu.:4.250  
 Max.   :1241232   Max.   :8.000   Max.   :8.000   Max.   :8.000  
       V5              V6              V7             V8              V9       
 Min.   :1.000   Min.   :1.000   Min.   :0.00   Min.   :1.000   Min.   : 1.00  
 1st Qu.:1.000   1st Qu.:2.000   1st Qu.:1.00   1st Qu.:2.000   1st Qu.: 1.00  
 Median :1.000   Median :2.000   Median :1.00   Median :2.500   Median : 1.00  
 Mean   :1.812   Mean   :2.438   Mean   :1.75   Mean   :3.125   Mean   : 2.75  
 3rd Qu.:1.000   3rd Qu.:2.000   3rd Qu.:1.00   3rd Qu.:3.250   3rd Qu.: 3.00  
 Max.   :9.000   Max.   :7.000   Max.   :7.00   Max

In [90]:
# imputing with regression with random noise

imputed_with_lm_noise<-Cancer_wisconsin # 1. Create a copy of the original dataset to impute with linear regression and random noise

fit_noise <- lm(V7 ~ V3 + V4 + V5 + V8 + V9 + V11, data = Cancer_wisconsin, subset = !is.na(V7)) # 2. fitting a linear regression model using the rows without missing values in column V7

imputed_with_lm_noise[is.na(imputed_with_lm_noise$V7),]

predicted_values_noise <- predict(fit_noise, newdata = imputed_with_lm_noise[is.na(imputed_with_lm_noise$V7),]) # 3. Predict the missing values using the fitted model

residuals <- fit_noise$residuals # 4. Get the residuals from the fitted model

# WE Need to be careful here 

noise <- rtruncnorm(n = length(predicted_values_noise),
                    a = 0,      # lower bound (0 => only positive)
                    b = 7,    # upper bound
                    mean = 0,
                    sd   = sd(residuals))
predicted_values_noise
imputed_with_lm_noise[is.na(imputed_with_lm_noise$V7), "V7"] <- as.integer(predicted_values_noise + noise) # 6. Impute the missing values in column V7 with the predicted values plus random noise

summary(imputed_with_lm_noise[index,])



,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11
,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>
24,1057013,8,4,5,1,2,NA,7,3,1,4
41,1096800,6,6,6,9,6,NA,7,8,1,2
140,1183246,1,1,1,1,1,NA,2,1,1,2
146,1184840,1,1,3,1,2,NA,2,1,1,2
159,1193683,1,1,2,1,3,NA,1,1,1,2
165,1197510,5,1,1,1,2,NA,3,1,1,2
236,1241232,3,1,4,1,2,NA,3,1,1,2
250,169356,3,1,1,1,2,NA,3,1,1,2
276,432809,3,1,3,1,2,NA,2,1,1,2


24        41       140       146       159       165       236       250 
7.2010468 3.2240912 1.2560440 1.6314383 1.2867756 1.4130095 1.9761009 1.4130095 
      276       293       295       298       316       322       412       618 
1.6314383 6.3054020 1.2560440 0.9567945 1.8316084 1.4130095 1.2560440 1.0990785

       V1                V2              V3              V4       
 Min.   :  61634   Min.   :1.000   Min.   :1.000   Min.   :1.000  
 1st Qu.: 595517   1st Qu.:1.000   1st Qu.:1.000   1st Qu.:1.000  
 Median :1057040   Median :3.000   Median :1.000   Median :2.500  
 Mean   : 857578   Mean   :3.375   Mean   :2.438   Mean   :2.875  
 3rd Qu.:1187051   3rd Qu.:5.000   3rd Qu.:4.000   3rd Qu.:4.250  
 Max.   :1241232   Max.   :8.000   Max.   :8.000   Max.   :8.000  
       V5              V6              V7             V8              V9       
 Min.   :1.000   Min.   :1.000   Min.   :1.00   Min.   :1.000   Min.   : 1.00  
 1st Qu.:1.000   1st Qu.:2.000   1st Qu.:2.00   1st Qu.:2.000   1st Qu.: 1.00  
 Median :1.000   Median :2.000   Median :3.00   Median :2.500   Median : 1.00  
 Mean   :1.812   Mean   :2.438   Mean   :3.00   Mean   :3.125   Mean   : 2.75  
 3rd Qu.:1.000   3rd Qu.:2.000   3rd Qu.:3.25   3rd Qu.:3.250   3rd Qu.: 3.00  
 Max.   :9.000   Max.   :7.000   Max.   :7.00   Max

In [91]:
# predict missing values with knn

imputed_with_knn<-Cancer_wisconsin # 1. Create a copy of the original dataset to impute with KNN

imputed_with_knn[is.na(imputed_with_knn$V7), "V7"] <- NA # Ensure that the missing values are set to NA

index_exclude<-which(is.na(imputed_with_knn$V7)) # Check the indices of the missing values in column V7

knn_model <- kknn(formula=V7 ~ ., train=imputed_with_knn[-index_exclude,],test= imputed_with_knn[index_exclude,], k = 5, distance = 2) # 2. Fit a KNN model using the rows without missing values in column V7

predicted_values_knn <- fitted(knn_model) # 3. Get the predicted values from the KNN model

prepredicted_values_knn <- as.integer(predicted_values_knn) # 4. Convert the predicted values to integers
length(prepredicted_values_knn)

length(imputed_with_lm[index_exclude,"V7"])
length(imputed_with_lm_noise[index_exclude,"V7"])


imputed_with_knn[index_exclude, "V7"] <- prepredicted_values_knn # 5. Impute the missing values in column V7 with the predicted values from the KNN model

Compare_imputations <- data.frame(
  knn          = as.factor(imputed_with_knn[index_exclude, "V7"]),
  lm           = as.factor(imputed_with_lm[index_exclude, "V7"]),
  lm_with_noise = as.factor(imputed_with_lm_noise[index_exclude, "V7"])
)
Compare_imputations
#colnames(Compare_imputations)<-c("KNN","LM","LM_with_noise") # Set the column names for the comparison data frame

#View(Compare_imputations)

[1] 16

[1] 16

[1] 16

knn,lm,lm_with_noise
<fct>,<fct>,<fct>
6,7,7
7,3,3
1,1,2
1,1,2
2,1,3
1,1,4
1,1,2
1,1,3
1,1,3
